In [ ]:
import json
import textwrap
# Load the JSON file
file_path = "course_data.json"  

with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# Define target chunk size
TARGET_CHUNK_SIZE = 300

# Function to split text while keeping words intact
def split_text_fixed(text, max_size=TARGET_CHUNK_SIZE):
    return textwrap.wrap(text, max_size, break_long_words=False, replace_whitespace=False)

# Re-chunk the dataset
new_data = []
new_chunk_id = 1

for chunk in data:
    original_text = chunk["text"]  
    split_chunks = split_text_fixed(original_text, TARGET_CHUNK_SIZE)
    
    for i, new_chunk in enumerate(split_chunks):
        new_data.append({
            "chunk_id": new_chunk_id,
            "source_file": chunk.get("source_file", "unknown"),  # Keep source tracking
            "chunk_number": i + 1,
            "content": new_chunk
        })
        new_chunk_id += 1

# Save the new chunked data
new_file_path = "rechunked_data300sz.json"

with open(new_file_path, "w", encoding="utf-8") as f:
    json.dump(new_data, f, indent=4)

print(f"Total chunks created: {len(new_data)}")

Total chunks created: 1580


In [27]:
import json
import re
from unicodedata import normalize

def clean_text(text, chunk_id):
    text = normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8')
    text = text.replace(' ?', ' = ?')
    lines = text.split('\n')
    seen = set()
    text = '\n'.join(line for line in lines if not (line.strip() in seen and seen.add(line.strip())))
    if "(truncated" in text:
        text = text.split("(truncated")[0].strip() + " [TRUNCATED - Content Incomplete]"
    text = re.sub(r'\s+', ' ', text).strip()
    paragraphs = text.split('\n')
    text = '\n'.join(para.strip() for para in paragraphs if para.strip())
    text = re.sub(r'[^\w\s.,;:!?\'"-/()]+', '', text)
    if chunk_id in [1, 2, 3, 4, 14, 17]:
        text = re.sub(r'Deadline:\s*(\w+\s*\d+(?:th|st|nd|rd)?,\s*\d{4})', r'\nDeadline: \1\n', text)
        text = text.replace('com- bines', 'combines')
    return text

def clean_json_file(input_data):
    cleaned_data = []
    for chunk in input_data:
        cleaned_chunk = chunk.copy()
        cleaned_chunk['content'] = clean_text(chunk['content'], chunk['chunk_id'])
        cleaned_data.append(cleaned_chunk)
    return cleaned_data

# Read from file
with open('rechunked_data300sz.json', 'r', encoding='utf-8') as f:
    input_data = json.load(f)  # Use json.load() for files

cleaned_data = clean_json_file(input_data)

# Output cleaned JSON
with open('cleaned.json', 'w', encoding='utf-8') as f:
    json.dump(cleaned_data, f, indent=4, ensure_ascii=False)

print("Cleaning complete.")

Cleaning complete.


In [29]:
# Load the cleaned JSON file
with open('cleaned.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

# Extract content and metadata
documents = [chunk['content'] for chunk in data]
metadata = [{'chunk_id': chunk['chunk_id'], 'source_file': chunk['source_file']} for chunk in data]

print(f"Loaded {len(documents)} documents.")


Loaded 1580 documents.


In [30]:
from sentence_transformers import SentenceTransformer

# Load the embedding model
embedder = SentenceTransformer('multi-qa-MiniLM-L6-cos-v1')
# Generate embeddings for all documents
embeddings = embedder.encode(documents, convert_to_tensor=True, show_progress_bar=True)

print(f"Embeddings shape: {embeddings.shape}") 

Batches:   0%|          | 0/50 [00:00<?, ?it/s]

Embeddings shape: torch.Size([1580, 384])


In [31]:
import faiss
import numpy as np

# Convert embeddings to numpy array and normalize for cosine similarity calculation
embeddings_np = embeddings.cpu().numpy()
embeddings_np = embeddings_np / np.linalg.norm(embeddings_np, axis=1)[:, np.newaxis]
dimension = embeddings_np.shape[1]
index = faiss.IndexFlatIP(dimension)  # Cosine similarity
index.add(embeddings_np)

print(f"FAISS index built with {index.ntotal} vectors.")

FAISS index built with 1580 vectors.


In [32]:
# Example query
query = "When is the assignment 1 deadline?"  # More precise query
query_embedding = embedder.encode([query], convert_to_tensor=True).cpu().numpy()
query_embedding = query_embedding / np.linalg.norm(query_embedding)  # Normalize
k = 3  # Retrieve top 3
distances, indices = index.search(query_embedding, k)

print("\nRetrieved Documents:")
for i, (dist, idx) in enumerate(zip(distances[0], indices[0])):
    print(f"{i+1}. Chunk ID: {metadata[idx]['chunk_id']}, Source: {metadata[idx]['source_file']}")
    print(f"Distance: {dist:.4f}")
    print(f"Content: {documents[idx][:200]}...")
    print()


Retrieved Documents:
1. Chunk ID: 1341, Source: unknown
Distance: 0.6221
Content: of the due date. Projects/Assignments missed due to legitimate reasons must be completed later mutually agreed with the instructor....

2. Chunk ID: 10, Source: unknown
Distance: 0.6037
Content: Deadline: February 14th, 2025...

3. Chunk ID: 708, Source: unknown
Distance: 0.4843
Content: Project proposals will be posted this week. A2 due Saturday 11:59PM! We still recommend using Colab for the assignments; in case you run into trouble (e.g. you have exceeded Colab quota), Please conta...



In [43]:
from huggingface_hub import HfApi

# Initialize the API
api = HfApi()

# List models filtered by a specific task compatible with pipeline
task = "Text2Text Generation"  # Options: "text-classification", "question-answering", "summarization", etc.
models = api.list_models(task=task, sort="downloads", direction=-1, limit=10)  # Top 10 by downloads

# Print model names
print(f"Top 10 models for {task}:")
for model in models:
    print(f"- {model.modelId} (Downloads: {model.downloads})")

Top 10 models for Text2Text Generation:
- prometheus-eval/prometheus-vision-7b-v1.0 (Downloads: 403)
- IDEA-CCNL/Randeng-T5-784M-MultiTask-Chinese (Downloads: 346)
- prometheus-eval/prometheus-vision-13b-v1.0 (Downloads: 181)
- IDEA-CCNL/Randeng-T5-77M-MultiTask-Chinese (Downloads: 95)
- abdelhalim/Rec_Business_Names (Downloads: 87)
- zahemen9900/finsight-ai (Downloads: 70)
- Hezam/ArabicT5-news-classification-generation (Downloads: 63)
- Hezam/ArabicT5-49GB-small-classification-generation (Downloads: 46)
- somosnlp-hackathon-2022/es_text_neutralizer (Downloads: 31)
- IDEA-CCNL/Randeng-T5-Char-57M-MultiTask-Chinese (Downloads: 30)


# Response Generation Layer

In [ ]:
from transformers import pipeline

generator = pipeline("text-generation", model="gpt2")  # Replace with a better model if needed
retrieved_docs = [documents[idx] for idx in indices[0]]
context = " ".join(retrieved_docs)
prompt = f"""
You are an AI Teaching Assistant for the SEP 775 Natural Language Processing course. 
You're mirroring human teaching assistant to have a typical conversation with student to respond to thier inquiries and based on the given course material (knowledge based), focusing on key concepts, instructions, or actionable information relevant to students or instructors.

Respond based on the material that is provided by the student which will be stored in the knowledge base. 

Notes to be aware off:
1. Always generate the answer based on the knowledge base.
2. You can extract an information from the knowledge base in the following conditions:
   - If the user is asking for a deadline (Project, Assignments), answer it using the date deadline that is stored in the correct chunk in the knowledge base.
   - If the user is asking when is the date of the quiz, answer it using the date that is stored in the knowledge base.

Based on this context: '{context}', answer the query: '{query}'"""
response = generator(prompt, max_length=1000, num_return_sequences=1)[0]["generated_text"]
print(response)

Device set to use cuda:0
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In [23]:
def search_and_respond(query, embedder, index, documents, metadata, generator):
    query_embedding = embedder.encode([query], convert_to_tensor=True).cpu().numpy()
    query_embedding = query_embedding / np.linalg.norm(query_embedding)
    distances, indices = index.search(query_embedding, k=3)
    retrieved_docs = [documents[idx] for idx in indices[0]]
    context = " ".join(retrieved_docs)
    prompt = f"""
You are an AI Teaching Assistant for the SEP 775 Natural Language Processing course. 
You're mirroring human teaching assistant to have a typical conversation with student to respond to thier inquiries and  based on the given course material (knowledge based), focusing on key concepts, instructions, or actionable information relevant to students or instructors.

Respond based on the material that is provided by the student which will be stored in the knowledge base. "GENERATE YOUR ANSWER BASED ON THE KNOWLEDGE BASE NOT JUST EXTRACTING"
Based on this context: '{context}', answer the query: '{query}'"""
    response = generator(prompt, max_length=1000)[0]["generated_text"]
    return response

while True:
    query = input("Enter your query (or 'exit' to quit): ")
    if query.lower() == "exit":
        break
    response = search_and_respond(query, embedder, index, documents, metadata, generator)
    print(f"Answer: {response}\n")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Answer: 
You are an AI Teaching Assistant for the SEP 775 Natural Language Processing course. 
You're mirroring human teaching assistant to have a typical conversation with student to respond to thier inquiries and  based on the given course material (knowledge based), focusing on key concepts, instructions, or actionable information relevant to students or instructors.

Respond based on the material that is provided by the student which will be stored in the knowledge base. "GENERATE YOUR ANSWER BASED ON THE KNOWLEDGE BASE NOT JUST EXTRACTING"
Based on this context: 'of the due date. Projects/Assignments missed due to legitimate reasons must be completed later mutually agreed with the instructor. Deadline: February 14th, 2025 Project proposals will be posted this week. A2 due Saturday 11:59PM! We still recommend using Colab for the assignments; in case you run into trouble (e.g. you have exceeded Colab quota), Please contact us 2', answer the query: 'what is the assignment 1 deadline?